> ## SOLUTION / ANSWER KEY &mdash; Lab 5.4
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-04-react-step-parsing.ipynb`](../lab-04-react-step-parsing.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 5.4 &mdash; Parsing a ReAct Step

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 3 &middot; Module 5 &mdash; What is Agentic AI?**

### What you'll do
- Read the ReAct text format a real model produces
- Extract the Action name and its Action Input
- Detect when a step is a Final Answer

> **How this lab works (near-real):** you have a local Ollama running `llama3.1:8b`. In Module 5 you build the agent **from scratch** &mdash; the loop, the ReAct parser, the tool router, the memory, the guardrails &mdash; as **real Python**. What's new vs the old version: a **real local model** now does the *reasoning* (it emits the ReAct steps / picks the actions) while **your** code parses, routes and loops it, and **real tools** run. Read the **Concept**, fill the real `___` blanks in **Build it**, then **Run it for real** and **read the trace**. Finish with an open **Your turn**. There is **no auto-grader**.

> **Framework note:** these labs use a **real local model** (`ollama run llama3.1:8b`, pinned to `http://127.0.0.1:11434`) via `langchain-ollama`. Unlike Module 6, you do **not** hand the loop to a framework &mdash; you build it yourself and the model drives it. If Ollama is down, the run cells print how to start it instead of crashing. A tool must **catch its own errors and return a string** &mdash; a tool that *raises* can abort the whole loop (you'll see exactly this in Labs 5 and 8). A small model can pick a wrong tool or fumble a step now and then &mdash; that's real agent behaviour, and it's why you read the trace.

**Reference:** [Module 5 slides &mdash; Tools &amp; function-calling &mdash; a ReAct trace](../../presentation/day3-module5-what-is-agentic-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 5 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))   # GROQ/OPENAI keys, if you ever want a hosted alternative

WORK = os.path.join(os.environ.get("TEMP") or os.environ.get("TMP") or "/tmp", "biaa-lab-05-04")
os.makedirs(WORK, exist_ok=True)

def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening. If it's down, start it with:  ollama run llama3.1:8b"""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False

from langchain_ollama import ChatOllama
# Day-3 provider: a REAL local model DRIVES the agent YOU build from scratch.
# Pin the host -- plain 'localhost' can give 'No route to host'.
llm = ChatOllama(model="llama3.1:8b", temperature=0, base_url="http://127.0.0.1:11434")

def llm_text(prompt):
    """Call the REAL model and return its text (the .content of the reply)."""
    return llm.invoke(prompt).content

if ollama_up():
    print("Ollama reachable at 127.0.0.1:11434 | model:", llm.model, "| WORK:", WORK)
else:
    print("Ollama NOT reachable -- start it with:  ollama run llama3.1:8b")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
A ReAct agent's LLM emits **structured text**: a `Thought:`, then either an `Action:` + `Action Input:`
(call a tool) or a `Final Answer:` (stop). The orchestrator must **parse** that text to know what to do
next. You'll build the parser on a fixed sample first (so you can test it with no model), then point it at
**real** `llama3.1:8b` output &mdash; the exact thing a LangChain output parser does under the hood.

In [ ]:
# DEMO -- the two kinds of step an agent produces
SAMPLE = """Thought: I need the population, so I should look it up.
Action: lookup
Action Input: population of metropolis"""

FINAL = """Thought: I now have everything I need.
Final Answer: 120000"""
print(SAMPLE); print("---"); print(FINAL)

## Build it
Write `field(text, label)` to read the text after a `Label:` line, then `is_final` to detect a Final Answer.
Test them on the fixed strings first.

In [ ]:
SAMPLE = """Thought: I need the population, so I should look it up.
Action: lookup
Action Input: population of metropolis"""

FINAL = """Thought: I now have everything I need.
Final Answer: 120000"""

def field(text, label):
    for line in text.splitlines():
        if line.strip().lower().startswith(label.lower() + ":"):
            return line.split(":", 1)[1].strip()
    return None

def is_final(text):
    return field(text, "Final Answer") is not None

try:
    print("action      =", field(SAMPLE, "Action"))
    print("action input=", field(SAMPLE, "Action Input"))
    print("SAMPLE final?", is_final(SAMPLE), "| FINAL final?", is_final(FINAL))
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## Run it for real &amp; read the trace
Now ask the **real** model to emit a ReAct step and parse *its* output with your `field` / `is_final`.

_This calls the real `llama3.1:8b` via Ollama. If Ollama is down the cell prints how to start it instead of crashing._

In [ ]:
if not ollama_up():
    print("Ollama not reachable -- start `ollama run llama3.1:8b`, then re-run this cell.")
else:
    try:
        spec = ("Reply in EXACTLY this format and nothing else:\n"
                "Thought: <one line of reasoning>\n"
                "Action: lookup\n"
                "Action Input: <the key>\n\n"
                "Task: find the population of metropolis.")
        text = llm_text(spec)                 # REAL model output -- your parser reads it
        print("MODEL EMITTED:\n" + text)
        print("---")
        print("parsed Action      :", field(text, "Action"))
        print("parsed Action Input:", field(text, "Action Input"))
        print("is it a final answer?:", is_final(text))
    except Exception as e:
        print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- Your parser turned the **real** model's free text into a structured `(Action, Action Input)` step &mdash; exactly what an output parser does.
- `field("Action")` must not accidentally grab `Action Input` &mdash; the `+ ":"` match is what keeps them apart.
- Real models sometimes add stray lines; a tolerant parser (first match wins, returns `None` if absent) survives that instead of crashing.

## Your turn (open task &mdash; no grader)
Change `spec` to nudge the model toward a **Final Answer** (e.g. *"you already know it is 120000"*) and
re-run. **What good looks like:** `is_final` flips to `True` and `field(text, "Final Answer")` returns the
number. Try a deliberately messy instruction and confirm your parser still doesn't crash.

In [ ]:
# --- Reference answer (ONE good way to do the 'Your turn' task -- compare with your own) ---
if ollama_up():
    spec = ("Reply in EXACTLY this format and nothing else:\n"
            "Thought: <one line>\nFinal Answer: <the number>\n\n"
            "Task: the population of metropolis is 120000 -- report it.")
    text = llm_text(spec)                 # nudged toward a Final Answer
    print("MODEL EMITTED:\n" + text)
    print("is_final?", is_final(text), "| Final Answer:", field(text, "Final Answer"))
else:
    print("(start Ollama to parse a real Final Answer)")

---
### Key takeaway
The orchestrator turns the model's text into structured steps. You parsed REAL llama3.1:8b output -- now you know exactly what LangChain's output parsers are reading.

[&#8592; All Module 5 labs](./index.html) &nbsp;&middot;&nbsp; [Module 5 slides](../../presentation/day3-module5-what-is-agentic-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>